This notebook follows this RADEK OSMULSKI's [notebook](https://www.kaggle.com/code/radek1/new-dataset-deberta-v3-large-training). If this is useful for you, please upvote for his great work.

As **CV** is very important in evaluating our model's performance, I'll establish a **CV** using KFold and train my model with the extra 500 samples in the notebook.

# Preprocessing the dataset

In [1]:
from typing import Optional, Union
import pandas as pd
import numpy as np
from colorama import Fore, Back, Style
from tqdm.notebook import tqdm
import torch
from datasets import Dataset
import gc
from dataclasses import dataclass
from transformers import AutoTokenizer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer, AutoModel, EarlyStoppingCallback
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer


deberta_v3_large = '/kaggle/input/deberta-v3-large-hf-weights'

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

We begin by loading and processing the train data.

In [2]:
df_train = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/train.csv')
df_train = df_train.drop(columns="id")
df_train.shape

(200, 7)

Let's add another 500 examples to the train set!

In [3]:
df_train = pd.concat([
    df_train,
    pd.read_csv('/kaggle/input/additional-train-data-for-llm-science-exam/extra_train_set.csv'),
])
df_train.reset_index(inplace=True, drop=True)
df_train.shape

(700, 7)

Now that we have gone from 200 -> 700 train examples, let us preprocess the data and begin training.

In [4]:
# cleaning the data by getting rid of weird characters

df_train_clean = df_train.replace('[^ -~]+', '', regex=True) # clean data; get rid of weird characters

df_train_clean = df_train_clean.replace('_', '', regex=True) # remove underscores
df_train_clean = df_train_clean.replace('\d+', '', regex=True) # remove numbers

df_train_clean

,prompt,A,B,C,D,E,answer
0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...,D
1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,A
2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...,A
3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,C
4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,D
...,...,...,...,...,...,...,...
695,How many different teams has Justis Logan Morr...,Seven,Nine,Ten,Five,Three,D
696,What is the significance of the cast iron wate...,The cast iron water pump is a remnant from the...,The cast iron water pump has no historical sig...,The cast iron water pump was installed on the ...,The cast iron water pump served as a decorativ...,The cast iron water pump was a functional tool...,C
697,What is the main characteristic of Cladosporiu...,Its spores are not the usual cause of seasonal...,It is able to grow under low water conditions ...,It can only grow in locations with high water ...,It only attacks the leaves of plants while spa...,It is classified as a bacteria that causes inv...,B
698,What influence is noted in the work of Francho...,Elaut's work shows the influence of Abstract E...,Elaut's work shows the influence of other Haar...,Elaut's work shows the influence of French Imp...,Elaut's work shows the influence of Flemish Ba...,Elaut's work shows the influence of Italian Re...,B


In [5]:
# vectorizing the data
# ngram currently unigram, can play around later
n=1
ngram_range=(n, n)
vectorizer = TfidfVectorizer(ngram_range=ngram_range)
X = vectorizer.fit_transform(df_train_clean)

tfidf_matrix = pd.DataFrame(X.toarray(), columns = vectorizer.get_feature_names_out())
df_train = df_train_clean
df_train

,prompt,A,B,C,D,E,answer
0,Which of the following statements accurately d...,MOND is a theory that reduces the observed mis...,MOND is a theory that increases the discrepanc...,MOND is a theory that explains the missing bar...,MOND is a theory that reduces the discrepancy ...,MOND is a theory that eliminates the observed ...,D
1,Which of the following is an accurate definiti...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,Dynamic scaling refers to the non-evolution of...,Dynamic scaling refers to the evolution of sel...,A
2,Which of the following statements accurately d...,The triskeles symbol was reconstructed as a fe...,The triskeles symbol is a representation of th...,The triskeles symbol is a representation of a ...,The triskeles symbol represents three interloc...,The triskeles symbol is a representation of th...,A
3,What is the significance of regularization in ...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,Regularizing the mass-energy of an electron wi...,C
4,Which of the following statements accurately d...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,The angular spacing of features in the diffrac...,D
...,...,...,...,...,...,...,...
695,How many different teams has Justis Logan Morr...,Seven,Nine,Ten,Five,Three,D
696,What is the significance of the cast iron wate...,The cast iron water pump is a remnant from the...,The cast iron water pump has no historical sig...,The cast iron water pump was installed on the ...,The cast iron water pump served as a decorativ...,The cast iron water pump was a functional tool...,C
697,What is the main characteristic of Cladosporiu...,Its spores are not the usual cause of seasonal...,It is able to grow under low water conditions ...,It can only grow in locations with high water ...,It only attacks the leaves of plants while spa...,It is classified as a bacteria that causes inv...,B
698,What influence is noted in the work of Francho...,Elaut's work shows the influence of Abstract E...,Elaut's work shows the influence of other Haar...,Elaut's work shows the influence of French Imp...,Elaut's work shows the influence of Flemish Ba...,Elaut's work shows the influence of Italian Re...,B


In [6]:
df_train = pd.concat([
    df_train,
    pd.read_csv('/kaggle/input/additional-train-data-for-llm-science-exam/extra_train_set.csv'),
])
df_train.reset_index(inplace=True, drop=True)
df_train.shape

(1200, 7)

In [7]:
option_to_index = {option: idx for idx, option in enumerate('ABCDE')}
index_to_option = {v: k for k,v in option_to_index.items()}

def preprocess(example):
    first_sentence = [example['prompt']] * 5
    second_sentences = [example[option] for option in 'ABCDE']
    tokenized_example = tokenizer(first_sentence, second_sentences, truncation=True)
    tokenized_example['label'] = option_to_index[example['answer']]
    
    return tokenized_example

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    
    def __call__(self, features):
        label_name = 'label' if 'label' in features[0].keys() else 'labels'
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]['input_ids'])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch['labels'] = torch.tensor(labels, dtype=torch.int64)
        return batch

Here we define a MAP@3 metric.

In [8]:
def map3(y_true, y_pred):
    m = (y_true.reshape((-1,1)) == y_pred)
    return np.mean(np.where(m.any(axis=1), m.argmax(axis=1)+1, np.inf)**(-1))

In [9]:
tokenizer = AutoTokenizer.from_pretrained(deberta_v3_large)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
/opt/conda/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:454: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Load the test data.

In [10]:
test_df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
test_df['answer'] = 'A'

test_ds = Dataset.from_pandas(test_df)
tokenized_test_ds = test_ds.map(preprocess, batched=False, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])

  0%|          | 0/200 [00:00<?, ?ex/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


# Training

In [11]:
%%time

final_dfs = pd.DataFrame()
kf = KFold(n_splits=4, shuffle=True, random_state=42)
cv_list = []
for fold, (idx_tr, idx_va) in enumerate(kf.split(df_train)):
    
    train_set = df_train.loc[idx_tr, ['prompt', 'A', 'B', 'C', 'D', 'E', 'answer']]
    valid_set = df_train.loc[idx_va, ['prompt', 'A', 'B', 'C', 'D', 'E', 'answer']]
    valid_label = df_train.loc[idx_va, 'answer'].values
    train_set = Dataset.from_pandas(train_set)
    tokenized_train = train_set.map(preprocess, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
    valid_set = Dataset.from_pandas(valid_set)
    tokenized_valid = valid_set.map(preprocess, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
    
    training_args = TrainingArguments(
        output_dir='./',
        overwrite_output_dir=True,
        load_best_model_at_end=True,
        save_total_limit=1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        warmup_ratio=0.8,
        learning_rate=2e-5,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=2,
        num_train_epochs=6,
        report_to='none',
        seed=42
    )
    model = AutoModelForMultipleChoice.from_pretrained(deberta_v3_large)

    trainer = Trainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
    )
    trainer.train()
    
    valid_pred = trainer.predict(tokenized_valid).predictions
    valid_pred_ids = np.argsort(-valid_pred, 1)
    valid_pred_letters = np.array(list('ABCDE'))[valid_pred_ids][:, :3]
    valid_map3 = map3(valid_label, valid_pred_letters)
    print(f"{Fore.RED}{Style.BRIGHT}Fold {fold}: MAP@3 = {valid_map3:.5f}{Style.RESET_ALL}")
    cv_list.append(valid_map3)
    
    test_predictions = trainer.predict(tokenized_test_ds).predictions
    fold_predict_df = pd.DataFrame(test_predictions, columns=[f'{x}{fold}' for x in ['A','B','C','D','E']])
    final_dfs = pd.concat([final_dfs, fold_predict_df], axis=1)
    
    del model, trainer, tokenized_train, tokenized_valid, train_set, valid_set
    gc.collect()

  0%|          | 0/900 [00:00<?, ?ex/s]

  0%|          | 0/300 [00:00<?, ?ex/s]

Some weights of the model checkpoint at /kaggle/input/deberta-v3-large-hf-weights were not used when initializing DebertaV2ForMultipleChoice: ['lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.dense.weight', 'lm_predictions.lm_head.bias', 'mask_predictions.dense.bias', 'mask_predictions.classifier.weight', 'lm_predictions.lm_head.LayerNorm.weight', 'mask_predictions.LayerNorm.weight', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.LayerNorm.bias', 'mask_predictions.classifier.bias', 'mask_predictions.dense.weight']
- This IS expected if you are initializing DebertaV2ForMultipleChoice from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2ForMultipleChoice from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertFor

Epoch,Training Loss,Validation Loss
1,1.606800,1.456147
2,1.146600,0.826041
3,0.491700,0.710000
4,0.314800,0.912398


Fold 0: MAP@3 = 0.84111


...

  0%|          | 0/300 [00:00<?, ?ex/s]

Some weights of the model checkpoint at /kaggle/input/deberta-v3-large-hf-weights were not used when initializing DebertaV2ForMultipleChoice: ['lm_predictions.lm_head.dense.bias', 'lm_predictions.lm_head.dense.weight', 'lm_predictions.lm_head.bias', 'mask_predictions.dense.bias', 'mask_predictions.classifier.weight', 'lm_predictions.lm_head.LayerNorm.weight', 'mask_predictions.LayerNorm.weight', 'mask_predictions.LayerNorm.bias', 'lm_predictions.lm_head.LayerNorm.bias', 'mask_predictions.classifier.bias', 'mask_predictions.dense.weight']
- This IS expected if you are initializing DebertaV2ForMultipleChoice from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaV2ForMultipleChoice from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertFor

Epoch,Training Loss,Validation Loss
1,1.614400,1.377012
2,1.143300,0.931431
3,0.442200,1.097436


Fold 3: MAP@3 = 0.79556


CPU times: user 54min 44s, sys: 6min 22s, total: 1h 1min 6s
Wall time: 1h 3min 46s


In [12]:
cv = np.mean(cv_list)
print(f"{Fore.RED}{Style.BRIGHT}Global MAP@3 = {cv:.5f}{Style.RESET_ALL}")

Global MAP@3 = 0.81500


# Predicting on the test set

In [13]:
final_dfs.head()

,A0,B0,C0,D0,E0,A1,B1,C1,D1,E1,A2,B2,C2,D2,E2,A3,B3,C3,D3,E3
0,-13.161530,-12.284928,-11.849982,-8.329967,-10.253127,-5.728049,-3.736434,-5.427474,0.998788,-5.855851,-10.100619,-7.809076,-10.079190,-1.670243,-10.099883,-2.691967,1.278306,-3.362448,5.022905,-4.087327
1,1.586252,-1.285799,-7.805361,-5.814055,-1.200350,-3.676574,-4.419388,-5.444935,-5.346001,-4.763092,-8.102806,-9.031918,-9.850274,-9.659022,-9.795027,-1.618613,-0.734055,-3.460888,-3.170298,-2.857190
2,-4.469829,-11.842309,-6.801966,-12.632163,-8.444260,0.774112,-5.348051,-1.685150,-5.350432,-5.493458,-9.144894,-10.209849,-10.004997,-10.157174,-10.183351,4.519513,-0.661225,3.511887,-1.956559,0.488150
3,-8.425795,-9.929697,-8.028378,-9.200804,-9.497153,-5.859974,-5.908206,-4.741765,-6.026130,-5.916085,-9.949893,-9.441405,4.329572,-8.933631,-7.211614,-4.559534,-3.867702,2.716607,-4.755106,-4.330171
4,-6.917577,-8.271272,-6.330070,-1.804971,-6.742867,-1.054319,-2.203609,1.374719,3.741763,-1.874115,-7.803642,-8.991486,-8.144547,-9.522592,-9.300555,-4.834526,-4.245650,0.189563,1.323657,-4.828067


In [14]:
final_dfs['A'] = final_dfs[['A1','A2','A3','A0']].mean(axis=1)
final_dfs['B'] = final_dfs[['B1','B2','B3','B0']].mean(axis=1)
final_dfs['C'] = final_dfs[['C1','C2','C3','C0']].mean(axis=1)
final_dfs['D'] = final_dfs[['D1','D2','D3','D0']].mean(axis=1)
final_dfs['E'] = final_dfs[['E1','E2','E3','E0']].mean(axis=1)

final_dfs[['A', 'B', 'C', 'D', 'E']].head()

,A,B,C,D,E
0,-7.920542,-5.638033,-7.679773,-0.994629,-7.574047
1,-2.952935,-3.867790,-6.640364,-5.997344,-4.653915
2,-2.080274,-7.015359,-3.745057,-7.524082,-5.908230
3,-7.198799,-7.286752,-1.430991,-7.228918,-6.738756
4,-5.152515,-5.928004,-3.227584,-1.565535,-5.686401


In [15]:
predictions_as_ids = np.argsort(-test_predictions, 1)
predictions_as_answer_letters = np.array(list('ABCDE'))[predictions_as_ids]
test_df['prediction'] = [' '.join(row) for row in predictions_as_answer_letters[:, :3]]

In [16]:
submission = test_df[['id', 'prediction']]
submission.to_csv('submission.csv', index=False)

display(pd.read_csv('submission.csv').head())
display(pd.read_csv('submission.csv').tail())

,id,prediction
0,0,D B A
1,1,B A E
2,2,A C E
3,3,C B E
4,4,D C B


,id,prediction
195,195,C A B
196,196,B C A
197,197,B A D
198,198,D A C
199,199,C A D


* there might be several ways to improve result
    - using more extra data
    - preprocessing
    - more reasonable CV strategy
    - different models
    - ensemble methods